# everbar — Jupyter smoke tests

Each cell exercises one behavior. Run top-to-bottom and eyeball the bars.

In [ ]:
import time

from everbar import Progress, detect_environment

detect_environment()

## 1. Iterator form, known total

In [ ]:
for i in Progress(range(10), desc="Iterator"):
    time.sleep(0.05)

## 2. Iterator form, unknown total (generator)

In [ ]:
def gen():
    for i in range(8):
        yield i

for x in Progress(gen(), desc="Unknown total"):
    time.sleep(0.05)

## 3. Context manager, manual update

In [ ]:
with Progress(total=10, desc="Manual") as bar:
    for i in range(10):
        time.sleep(0.05)
        bar.update(1)

## 4. set_postfix with numbers

In [ ]:
with Progress(total=10, desc="Training") as bar:
    for i in range(10):
        time.sleep(0.05)
        bar.set_postfix(loss=1.0 / (i + 1), step=i)
        bar.update(1)

## 5. set_postfix with list (Rich-markup regression)

In [ ]:
with Progress(total=5, desc="With list") as bar:
    for i in range(5):
        time.sleep(0.05)
        bar.set_postfix(values=[1, 2, 3], i=i)
        bar.update(1)

## 6. Nested bars

In [ ]:
for epoch in Progress(range(3), desc="Epoch"):
    for step in Progress(range(5), desc=f"  step (epoch {epoch})"):
        time.sleep(0.02)

## 7. Exception inside context manager (should not leave the bar hanging)

In [ ]:
try:
    with Progress(total=10, desc="Will fail") as bar:
        for i in range(10):
            time.sleep(0.05)
            if i == 5:
                raise RuntimeError("boom")
            bar.update(1)
except RuntimeError as e:
    print(f"caught: {e}")

## 8. disable=True (silent)

In [ ]:
out = list(Progress(range(5), desc="Silent", disable=True))
assert out == [0, 1, 2, 3, 4]
"ok"

## 9. Failure state (one worker errors, job continues)

In [ ]:
with Progress(total=10, desc="Batch") as bar:
    for i in range(10):
        time.sleep(0.05)
        if i == 5:
            bar.fail()
        bar.update(1)